# 05 Clustering Resolution — проверка fusion-графа

Эта тетрадка читает результаты `04_fusion_pack_grouping.ipynb` и проверяет, насколько graph resolution совпадает с размеченным gold-set на уровне связей.


## Как читать метрики

`precision` показывает, какая доля предсказанных связей действительно правильная. `recall` показывает, какую долю настоящих связей мы нашли. Для family false link опаснее, потому что может склеить разные товары в один компонент.


## Блок кода 1. Подготовка окружения


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "research" / "dedup").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


## Блок кода 2. Загрузка fusion artifacts


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
COMPONENTS_PATH = DATA_DIR / "fusion_components_sauces.csv"
PAIR_EVAL_PATH = DATA_DIR / "fusion_pair_eval_sauces.csv"
EVAL_SPLIT = os.environ.get("DEDUP_FUSION_EVAL_SPLIT", "test")

if not COMPONENTS_PATH.exists() or not PAIR_EVAL_PATH.exists():
    raise FileNotFoundError(
        "Run notebooks/04_fusion_pack_grouping.ipynb first: expected "
        f"{COMPONENTS_PATH.name} and {PAIR_EVAL_PATH.name}."
    )

components = pd.read_csv(COMPONENTS_PATH)
pair_eval = pd.read_csv(PAIR_EVAL_PATH)

print(f"Loaded components: {len(components)} rows")
print(f"Loaded pair eval: {len(pair_eval)} rows")
if {"fusion_method", "fusion_threshold_strategy", "fusion_threshold_same"}.issubset(pair_eval.columns):
    display(pair_eval[["fusion_method", "fusion_threshold_strategy", "fusion_threshold_same"]].drop_duplicates())


## Блок кода 3. Link-level качество family и pack


In [ ]:
def _as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


def _binary_link_report(frame: pd.DataFrame, *, true_col: str, pred_col: str, scope: str) -> dict[str, object]:
    true_link = _as_bool(frame[true_col])
    pred_link = _as_bool(frame[pred_col])
    tp = int((true_link & pred_link).sum())
    fp = int((~true_link & pred_link).sum())
    fn = int((true_link & ~pred_link).sum())
    tn = int((~true_link & ~pred_link).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "scope": scope,
        "eval_split": frame["split"].iloc[0] if "split" in frame.columns and frame["split"].nunique() == 1 else "all",
        "pairs": len(frame),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positive_links": tp,
        "false_links": fp,
        "missed_links": fn,
        "true_negative_links": tn,
    }


eval_pairs = pair_eval[pair_eval["split"].astype(str).eq(EVAL_SPLIT)].copy() if "split" in pair_eval.columns else pair_eval.copy()
if eval_pairs.empty:
    eval_pairs = pair_eval.copy()

link_report = pd.DataFrame(
    [
        _binary_link_report(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", scope="family"),
        _binary_link_report(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", scope="pack"),
    ]
)
display(link_report)


## Блок кода 4. Размеры компонентов

Здесь важно быстро заметить огромные компоненты: они часто означают chaining из-за false merge.


In [ ]:
def _component_summary(frame: pd.DataFrame, component_col: str, graph: str) -> dict[str, object]:
    if component_col not in frame.columns or frame.empty:
        return {"graph": graph, "components": 0, "multi_node_components": 0, "max_nodes": 0}
    sizes = frame.groupby(component_col, dropna=True).size().rename("nodes").reset_index()
    return {
        "graph": graph,
        "components": int(sizes[component_col].nunique()),
        "multi_node_components": int((sizes["nodes"] > 1).sum()),
        "max_nodes": int(sizes["nodes"].max()) if not sizes.empty else 0,
    }


component_report = pd.DataFrame(
    [
        _component_summary(components, "fusion_family_id", "pred_family"),
        _component_summary(components, "fusion_pack_id", "pred_pack"),
        _component_summary(components, "true_family_id", "true_family_partial"),
        _component_summary(components, "true_pack_id", "true_pack_partial"),
    ]
)
display(component_report)

for component_col in ["fusion_family_id", "fusion_pack_id"]:
    if component_col in components.columns:
        sizes = components.groupby(component_col, dropna=True).size().rename("nodes").reset_index()
        display(sizes.sort_values("nodes", ascending=False).head(15))


## Блок кода 5. Примеры ошибок графа


In [ ]:
def _show_examples(frame: pd.DataFrame, *, true_col: str, pred_col: str, title: str) -> None:
    true_link = _as_bool(frame[true_col])
    pred_link = _as_bool(frame[pred_col])
    cols = [
        "split",
        "score",
        "same_base_product",
        "same_pack_signature",
        "title_a",
        "title_b",
        "brand_a",
        "brand_b",
        "unit_amount_a",
        "unit_amount_b",
        "total_amount_a",
        "total_amount_b",
        "multipack_count_a",
        "multipack_count_b",
    ]
    cols = [column for column in cols if column in frame.columns]
    print(f"\n{title}: false links")
    display(frame[~true_link & pred_link][cols].head(10))
    print(f"\n{title}: missed links")
    display(frame[true_link & ~pred_link][cols].head(10))


_show_examples(eval_pairs, true_col="true_same_family", pred_col="pred_same_family", title="Family")
_show_examples(eval_pairs, true_col="true_same_pack", pred_col="pred_same_pack", title="Pack")


## Что делать после этой тетрадки

Если family false links выглядят опасно, возвращаемся в `03` и выбираем более строгий method/strategy. Если family нормальная, но pack странный, проверяем deterministic weight/multipack поля.
